In [1]:
import os
import sys
import datetime
import pandas as pd
import time
import traceback
import _pickle as pickle

# upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
upper_dir = '/home/trade_core' # TEST
sys.path.append(upper_dir)
from loggers.logger import KimpBotLogger
from exchange_plugin.upbit_plug import UserUpbitAdaptor
from exchange_plugin.binance_plug import UserBinanceAdaptor
from etc.redis_connector.redis_connector import InitRedis
from exchange_plugin.integrated_plug import UserExchangeAdaptor
from api.utils import decrypt_data
from etc.db_handler.postgres_client import InitDBClient



In [2]:
import json

config_dir = "/home/trade_core/trade_core_config.json"
logging_dir = "/home/trade_core/exchange_plugin"

with open(config_dir) as f:
    config = json.load(f)
if not os.path.exists(logging_dir):
    os.mkdir(logging_dir)
# if node not in config['node_settings'].keys():
#     raise Exception(f"Node name should be the one of {list(config['node_settings'].keys())}")
node = config['node']
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_telegram_id_list = []
admin_telegram_id = config['telegram_admin_id']['charlie1155']
admin_telegram_id_list.append(admin_telegram_id)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
db_dict = config['database_setting'][config['node_settings'][node]['db_settings']]

In [3]:
db_client = InitDBClient(**{**db_dict, "database":'trade_core'})

InitDBClient|SCHEMA: trade_core already exists.


In [4]:
# Fetch api keys
conn = db_client.get_conn()
sql = f"SELECT * FROM exchange_api_key WHERE trade_config_uuid = %s AND market_code = %s"
val = ("e16b3bdc-301b-4512-be3b-f0e60cef7004", "BINANCE_USD_M/USDT")

curr = conn.cursor()
curr.execute(sql, val)
api_keys = curr.fetchall()
conn.close()

In [5]:
binance_access_key = decrypt_data(bytes(api_keys[0][9])).decode()
binance_secret_key = decrypt_data(bytes(api_keys[0][10])).decode()

upbit_access_key = "x2AKLfsRtAgRxFjQ7p9TZTAg6E1SveoqfNfD8MK8"
upbit_secret_key = "svEran52QdsUyJdxAPYoTgFT2MXsHc5ZiKsKPxTL"

okx_access_key = "fcb66a6e-0443-4743-8d9b-61caf7eab662"
okx_secret_key = "0029E09874B38F5AC7E68E9DFC348667"
okx_passphrase = "121431Rn!!"

bybit_access_key = "S3YKBbD0kz1WwcfqZD"
bybit_secret_key = "390M6dKJ9J0uEK7vXYAl3qxVCh944tRrzHah"


In [6]:

trade_df_dict = {}
market_code_combination = "UPBIT_SPOT/KRW:BINANCE_USD_M/USDT"
get_premium_df = lambda x: x

user_exchange_adaptor = UserExchangeAdaptor(admin_telegram_id=admin_telegram_id, node='TEST', db_dict=db_dict, trade_df_dict=trade_df_dict, market_code_combination=market_code_combination, get_premium_df=get_premium_df, logging_dir=logging_dir)

2024-05-24 04:48:30,299 - integrated_plug_UPBIT_SPOT__KRW-BINANCE_USD_M__USDT - INFO - integrated_plug_UPBIT_SPOT/KRW:BINANCE_USD_M/USDT_logger init
2024-05-24 04:48:30,301 - integrated_plug_UPBIT_SPOT__KRW-BINANCE_USD_M__USDT - INFO - handle_trade_info_queue_loop started.
2024-05-24 04:48:30,303 - integrated_plug_UPBIT_SPOT__KRW-BINANCE_USD_M__USDT - INFO - handle_margin_liquidation_call_trade_queue_loop started.
2024-05-24 04:48:30,333 - user_upbit_plug_UPBIT_SPOT__KRW - INFO - user_upbit_plug_UPBIT_SPOT/KRW started.
2024-05-24 04:48:30,336 - user_upbit_plug_UPBIT_SPOT__KRW - INFO - loop_load_user_api_keys started.
2024-05-24 04:48:30,338 - user_upbit_plug_UPBIT_SPOT__KRW - INFO - handle_order_info_queue_loop started.


2024-05-24 04:48:30,360 - user_binance_plug_BINANCE_USD_M__USDT - INFO - user_binance_plug_BINANCE_USD_M/USDT started.
2024-05-24 04:48:30,361 - user_binance_plug_BINANCE_USD_M__USDT - INFO - loop_load_user_api_keys started.
2024-05-24 04:48:30,362 - user_binance_plug_BINANCE_USD_M__USDT - INFO - handle_order_info_queue_loop started.
2024-05-24 04:48:30,362 - user_binance_plug_BINANCE_USD_M__USDT - INFO - start_socket_stream|start_socket_stream started.
2024-05-24 04:48:30,368 - integrated_plug_UPBIT_SPOT__KRW-BINANCE_USD_M__USDT - INFO - load_order_history_loop started.
2024-05-24 04:48:30,369 - integrated_plug_UPBIT_SPOT__KRW-BINANCE_USD_M__USDT - INFO - load_trade_history_loop started.


InitDBClient|SCHEMA: trade_core already exists.
InitDBClient|SCHEMA: trade_core already exists.
InitDBClient|SCHEMA: trade_core already exists.


In [7]:
user_exchange_adaptor.change_margin_type('BINANCE', binance_access_key, binance_secret_key, 'USD_M', '', 'EOSUSDT', None)

2024-05-24 04:48:31,461 - user_binance_plug_BINANCE_USD_M__USDT - INFO - margin_type: CROSSED


In [ ]:
ticker_df = pickle.loads(user_exchange_adaptor.local_redis_client.get_data('TRADE_CORE|binance_usd_m_ticker_df'))
ticker_df

In [ ]:
capital_series = user_exchange_adaptor.get_capital('upbit', upbit_access_key, upbit_secret_key, 'SPOT')
capital_series

In [ ]:
capital_series = user_exchange_adaptor.get_capital('binance', binance_access_key, binance_secret_key, 'USD_M')
capital_series.to_json()

In [ ]:
client = user_exchange_adaptor.exchange_adaptor_dict['BINANCE'].load_user_client(binance_access_key, binance_secret_key)
client.futures_account()